# Tutorial 12 · Entropy, Huffman & KL — Part B

**~40 minutes · after the worksheet**

Compute surprise and entropy, build a Huffman code, and measure the excess cost of a wrong model.

Work in pairs. Before each code cell, write down the qualitative result you expect. The notebook is designed to run top-to-bottom in a fresh kernel.


## 1 · Entropy, cross-entropy, and KL


In [1]:
import math, heapq, itertools
import numpy as np

def entropy(p):
    p=np.asarray(p,float); return -np.sum(p[p>0]*np.log2(p[p>0]))
def cross_entropy(p,q): return -np.sum(np.asarray(p)*np.log2(q))
p=np.array([.75,.2,.05]); q=np.array([.45,.35,.2])
print("H",entropy(p),"CE",cross_entropy(p,q),"KL",cross_entropy(p,q)-entropy(p))


H 0.9917601481809735 CE 1.2830133593941073 KL 0.29125321121313386


## 2 · Build a deterministic Huffman code


In [2]:
weights={"A":40,"B":20,"C":15,"D":15,"E":10}; serial=itertools.count()
heap=[(w,next(serial),s) for s,w in weights.items()]; heapq.heapify(heap)
while len(heap)>1:
    wa,_,a=heapq.heappop(heap); wb,_,b=heapq.heappop(heap)
    heapq.heappush(heap,(wa+wb,next(serial),(a,b)))
tree=heap[0][2]
def codes(node,prefix="",out=None):
    out={} if out is None else out
    if isinstance(node,str): out[node]=prefix
    else: codes(node[0],prefix+"0",out); codes(node[1],prefix+"1",out)
    return out
codebook=codes(tree); print(codebook)
L=sum(weights[s]*len(codebook[s]) for s in weights)/sum(weights.values())
print("average length",L)


{'A': '0', 'E': '100', 'C': '101', 'D': '110', 'B': '111'}
average length 2.2


## 3 · Encode and decode


In [3]:
message="ABACADABRA".replace("R","E")
bits="".join(codebook[ch] for ch in message)
reverse={v:k for k,v in codebook.items()}; decoded=[]; prefix=""
for bit in bits:
    prefix+=bit
    if prefix in reverse: decoded.append(reverse[prefix]); prefix=""
print(message,bits,"".join(decoded),len(bits))


ABACADABEA 01110101011001111000 ABACADABEA 20


## Closing check

Write three sentences: one numerical result you verified, one geometric/probabilistic interpretation, and one failure mode you would now test in a larger implementation.
